2026-04-08

Shahlo Solieva

Notebook to calculate distances between ALL ATOMS on antibody to ALL ATOMS of antigen protein residues. Excluding hydrogen atoms. 

In [1]:
import numpy as np
import mdtraj as md
import matplotlib.pyplot as plt
import itertools
import pandas as pd

# Functions

In [2]:
def get_resis(sys_path, start_of_VH, end_of_VH, start_of_VL, end_of_VL):
    # define inputs directory path and print out all input variables.
    cluster_dir_path = f"/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/clustering/radius_15_variable/data/{sys_path}/"
    print("\tINPUTS:", sys_path,start_of_VH, end_of_VH, start_of_VL, end_of_VL)

    # select alpha carbons from the variable region:
    #pdb_path = f'{cluster_dir_path}prot_masses_uniform.pdb'
    pdb_path = f'{cluster_dir_path}prot_masses.pdb'
    pdb = md.load(pdb_path)
    all_ca_atoms = pdb.topology.select(f'chainid 0 and residue {start_of_VH} to {end_of_VH} and name CA or chainid 1 and residue {start_of_VL} to {end_of_VL} and name CA')

    # check to make sure that the atoms / atom indicies selected are from the correct residues.
    variable_resi_names = []
    for i in range(len(all_ca_atoms)):
        variable_resi_names.append(pdb.topology.atom(all_ca_atoms[i]).residue)
    variable_HC_LC_resi_names = [variable_resi_names[:end_of_VH], variable_resi_names[end_of_VH:]]
    return variable_HC_LC_resi_names

In [3]:
colors=['#80bfff','#004080']

def import_data_apo(system_name):
    base_path = '/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/per_resi_RMSD'
    rmsd_data = np.load(f'{base_path}/data/apo/{system_name}/VHC_VLC_CA_RMSD_per_resi_to_average_structure.npy', allow_pickle=True)
    return rmsd_data 

def get_HC_rmsd(rmsd_data, vh_end):
    return rmsd_data[:vh_end]*10
def get_LC_rmsd(rmsd_data, vh_end):
    return rmsd_data[vh_end:]*10

def calculate_min_dist_values(pdb_path, HC, LC, antigen_, vh_end, vl_end):
    '''
    examples: 
        antigen = '8 10'
        HC = '5' 
        LC = '1' 
    '''
    pdb = md.load(pdb_path)
    heavy_chain_CA = pdb.topology.select(f'chainid {HC} and name CA')
    light_chain_CA = pdb.topology.select(f'chainid {LC} and name CA')
    antigen = pdb.topology.select(f'chainid {antigen_} and not element H')
    print(len(heavy_chain_CA), len(light_chain_CA), len(antigen))
    HC_protein_minDists = []
    HC_resis = []
    for i in range(len(heavy_chain_CA)):
        Ab_residue_n = int(str(pdb.topology.atom(heavy_chain_CA[i]).residue)[3:]) 
        HC_resis.append(pdb.topology.atom(heavy_chain_CA[i]).residue)
        Ab_residue_atoms = pdb.topology.select(f'chainid {HC} and residue {Ab_residue_n} and not element H')
        HC_protein_pairs = list(itertools.product(Ab_residue_atoms, antigen))
        HC_protein_minDists.append(np.min(md.compute_distances(pdb, HC_protein_pairs)[0]))
    LC_protein_minDists = []
    LC_resis = []
    for i in range(len(light_chain_CA)):
        Ab_residue_n = int(str(pdb.topology.atom(light_chain_CA[i]).residue)[3:]) 
        LC_resis.append(pdb.topology.atom(light_chain_CA[i]).residue)
        Ab_residue_atoms = pdb.topology.select(f'chainid {LC} and residue {Ab_residue_n} and not element H')
        LC_protein_pairs = list(itertools.product(Ab_residue_atoms, antigen))
        LC_protein_minDists.append(np.min(md.compute_distances(pdb, LC_protein_pairs)[0]))
    HC_protein_minDists = np.array(HC_protein_minDists[:vh_end])*10 # nm to Angstrom, and only save VHC values. 
    LC_protein_minDists = np.array(LC_protein_minDists[:vl_end])*10 # nm to Angstrom, and only save VLC values.
    return HC_protein_minDists, LC_protein_minDists, HC_resis[:vh_end], LC_resis[:vl_end]


In [4]:
base_path='/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/distance_to_antigen/'

# CAP256

## import and calculate data. 

### Mature

In [5]:
# import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
print('Importing RMSD data for mature antibody')
system_name='CAP256_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 145, 1, 111 # Mature CAP256: "1" "145" "1" "111"
mature_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
print(len(rmsd_data))
mature_HC_rmsd_values, mature_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{base_path}structures/6vtt_cap256_clean_renumbered.pdb'
HC, LC, antigen = '3', '4', '0 1 2'
pdb_HC_protein_minDists, pdb_LC_protein_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)

Importing RMSD data for mature antibody
	INPUTS: apo/CAP256_mature_VC 1 145 1 111
256
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.


/home/ssolieva/miniconda/envs/pyemma/lib/python3.11/site-packages/mdtraj/formats/pdb/pdbfile.py:200: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn('Unlikely unit cell vectors detected in PDB file likely '


143 108 10258


#### alignment of pdb data

In [6]:
print(len(mature_VHC_VLC_resi_names[0]), len(mature_VHC_VLC_resi_names[1]))
print(len(pdb_HC_resis), len(pdb_LC_resis))
print(mature_VHC_VLC_resi_names[0][0:2], mature_VHC_VLC_resi_names[1][0:2])
print(pdb_HC_resis[0:2],pdb_LC_resis[0:2])
# add 'gln1' to mature HC

145 111
143 108
[GLN1, VAL2] [GLN1, SER2]
[VAL2, GLN3] [GLN1, SER2]


In [7]:
# Make sure everything is aligned:  
# residues list alignment
pdb_HC_resis_fixed = pdb_HC_resis[:]
pdb_HC_resis_fixed.insert(0,'GLN1')
# check that the mature sequences are aligned. 
for i in range(len(pdb_HC_resis_fixed)):
    if str(pdb_HC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[0][i]):
        print('ERROR: misalignment',i, pdb_HC_resis_fixed[i], mature_VHC_VLC_resi_names[0][i])

# minimum distance alignment
pdb_HC_protein_minDists_fixed = np.insert(pdb_HC_protein_minDists, 0, None) # to account for the added 'gln1' in HC resis. 

ERROR: misalignment 111 TYS112 TYR112
ERROR: misalignment 112 TYS113 TYR113


### naive

In [8]:
# Import RMSD data for the naive antibody now.
print('Importing RMSD data for naive antibody')
system_name='CAP256_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 144, 1, 111
naive_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
print(len(rmsd_data))
naive_HC_rmsd_values, naive_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end) #pdb_LC_protein_minDists

Importing RMSD data for naive antibody
	INPUTS: apo/CAP256_naive_VC 1 144 1 111
255


## Alignment

In [9]:
# Light chain for CAP256 is already aligned. Just save the output. 

In [10]:
# Heavy chain for CAP256 needs to be aligned. Adding to naive and mature. Naive needs 1 extra None. 
print(len(naive_HC_rmsd_values), len(mature_HC_rmsd_values))

# add 'None' as gaps for the sequence alignment
x0 = np.insert(naive_HC_rmsd_values, [118,118], None)
x1 = np.insert(x0, [122], None)
x2 = np.insert(x1, [128], None)
x3 = np.insert(x2, [133], None)
naive_HC_rmsd_values_fixed = x3

naive_VHC_resi_names_fixed___ = np.insert(naive_VHC_VLC_resi_names[0],   [118,118], None)
naive_VHC_resi_names_fixed__  = np.insert(naive_VHC_resi_names_fixed___, [122], None)
naive_VHC_resi_names_fixed_   = np.insert(naive_VHC_resi_names_fixed__,  [128], None)
naive_VHC_resi_names_fixed    = np.insert(naive_VHC_resi_names_fixed_,   [133], None)

y1 = np.insert(mature_HC_rmsd_values, [115,115], None)
y2 = np.insert(y1, [126], None)
y3 = np.insert(y2, [135], None)
mature_HC_rmsd_values_fixed = y3

mature_VHC_resi_names_fixed__ = np.insert(mature_VHC_VLC_resi_names[0],   [115,115], None)
mature_VHC_resi_names_fixed_  = np.insert(mature_VHC_resi_names_fixed__, [126], None)
mature_VHC_resi_names_fixed   = np.insert(mature_VHC_resi_names_fixed_,  [135], None)

print(len(naive_HC_rmsd_values_fixed), len(mature_HC_rmsd_values_fixed))
print(len(naive_VHC_resi_names_fixed), len(mature_VHC_resi_names_fixed))

# PDB data alignment:
z1 = np.insert(pdb_HC_resis_fixed, [115,115], None)
z2 = np.insert(z1, [126], None)
z3 = np.insert(z2, [135], None)
z4 = np.insert(z3, len(z3), None)
pdb_HC_resis_fixed_fixed = z4

pdb_HC_protein_minDists_fixed___ = np.insert(pdb_HC_protein_minDists_fixed,   [115,115], None)
pdb_HC_protein_minDists_fixed__  = np.insert(pdb_HC_protein_minDists_fixed___, [126], None)
pdb_HC_protein_minDists_fixed_   = np.insert(pdb_HC_protein_minDists_fixed__,  [135], None)
pdb_HC_protein_minDists_fixed_fixed   = np.insert(pdb_HC_protein_minDists_fixed_,  len(pdb_HC_protein_minDists_fixed_), None)

print(len(pdb_HC_resis_fixed_fixed), len(pdb_HC_protein_minDists_fixed_fixed))

pdb_LC_resis_fixed = np.insert(pdb_LC_resis, [108,108,108], None)
pdb_LC_protein_minDists_fixed  = np.insert(pdb_LC_protein_minDists, [108,108,108], None)
print(len(pdb_LC_resis_fixed), len(pdb_LC_protein_minDists_fixed))

144 145
149 149
149 149
149 149
111 111


## Save output for plotting later. 

In [11]:
# HC
data = {
    "pdb_mature_HC_resi":pdb_HC_resis_fixed_fixed,
    "pdb_mature_HC_protein_minDists":pdb_HC_protein_minDists_fixed_fixed,
    "naive_HC_resi": naive_VHC_resi_names_fixed,
    "naive_HC_rmsd": naive_HC_rmsd_values_fixed,
    "mature_HC_resi": mature_VHC_resi_names_fixed,
    "mature_HC_rmsd": mature_HC_rmsd_values_fixed
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/CAP256_HC.csv"
df.to_csv(output_filename, index=False)

# LC
data = {
    "pdb_mature_LC_resi":pdb_LC_resis_fixed,
    "pdb_mature_LC_protein_minDists":pdb_LC_protein_minDists_fixed,
    "naive_LC_resi": naive_VHC_VLC_resi_names[1],
    "naive_LC_rmsd": naive_LC_rmsd_values,
    "mature_LC_resi": mature_VHC_VLC_resi_names[1],
    "mature_LC_rmsd": mature_LC_rmsd_values,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/CAP256_LC.csv"
df.to_csv(output_filename, index=False)

# 1-18

## import and calculate data. 

### Mature

In [12]:
# import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
print('Importing RMSD data for mature antibody')
system_name='Ab118_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 131, 1, 108
mature_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
mature_HC_rmsd_values, mature_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{base_path}structures/6udj_1-18_clean_renumbered.pdb'
HC, LC, antigen = '5', '1', '8 10'
pdb_HC_protein_minDists, pdb_LC_protein_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)

Importing RMSD data for mature antibody
	INPUTS: apo/Ab118_mature_VC 1 131 1 108
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
130 108 6756


#### alignment of pdb data

In [13]:
# Make sure everything is aligned:  
# residues list alignment
pdb_HC_resis_fixed = pdb_HC_resis[:]
pdb_HC_resis_fixed.insert(0,'GLN1')
# check that the mature sequences are aligned. 
for i in range(len(pdb_HC_resis_fixed)):
    if str(pdb_HC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[0][i]):
        print('ERROR: misalignment',i, pdb_HC_resis_fixed[i], mature_VHC_VLC_resi_names[0][i])

# minimum distance alignment
pdb_HC_protein_minDists_fixed = np.insert(pdb_HC_protein_minDists, 0, None) # to account for the added 'gln1' in HC resis. 

### naive

In [14]:
# Import RMSD data for the naive antibody now.
print('Importing RMSD data for naive antibody')
system_name='Ab118_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 125, 1, 108
naive_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
naive_HC_rmsd_values, naive_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end) #pdb_LC_protein_minDists

Importing RMSD data for naive antibody
	INPUTS: apo/Ab118_naive_VC 1 125 1 108


## Alignment

In [15]:
# Light chain for 1-18 is already aligned. Just save the output. 

In [16]:
# Heavy chain for 1-18 needs to be aligned. Adding a total of 6 'None' values to naive sequence. 
print(len(naive_HC_rmsd_values), len(mature_HC_rmsd_values))
print(len(naive_LC_rmsd_values), len(mature_LC_rmsd_values))

# add 'None' as gaps for the sequence alignment
x0 = np.insert(naive_HC_rmsd_values, [26,26], None)
x1 = np.insert(x0, [29,29,29,29], None)
naive_HC_rmsd_values_fixed = x1

naive_VHC_resi_names_fixed_ = np.insert(naive_VHC_VLC_resi_names[0], [26,26], None)
naive_VHC_resi_names_fixed = np.insert(naive_VHC_resi_names_fixed_, [29,29,29,29], None)

print(len(naive_HC_rmsd_values_fixed), len(mature_HC_rmsd_values))
print(len(naive_LC_rmsd_values), len(mature_LC_rmsd_values)) #pdb_LC_protein_minDists

125 131
108 108
131 131
108 108


## Save output for plotting later. 

In [17]:
# HC
data = {
    "pdb_mature_HC_resi":pdb_HC_resis_fixed,
    "pdb_mature_HC_protein_minDists":pdb_HC_protein_minDists_fixed,
    "naive_HC_resi": naive_VHC_resi_names_fixed,
    "naive_HC_rmsd": naive_HC_rmsd_values_fixed,
    "mature_HC_resi": mature_VHC_VLC_resi_names[0],
    "mature_HC_rmsd": mature_HC_rmsd_values,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/Ab118_HC.csv"
df.to_csv(output_filename, index=False)

# LC
data = {
    "pdb_mature_LC_resi":pdb_LC_resis,
    "pdb_mature_LC_protein_minDists":pdb_LC_protein_minDists,
    "naive_LC_resi": naive_VHC_VLC_resi_names[1],
    "naive_LC_rmsd": naive_LC_rmsd_values,
    "mature_LC_resi": mature_VHC_VLC_resi_names[1],
    "mature_LC_rmsd": mature_LC_rmsd_values,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/Ab118_LC.csv"
df.to_csv(output_filename, index=False)

# CH103

## Import and Calculate data

### Mature

In [18]:
# import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
print('Importing RMSD data for mature antibody')
system_name='CH103_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 121, 1, 105 # Mature 
mature_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
mature_HC_rmsd_values, mature_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{base_path}structures/9d7o_CH103_clean_renumbered.pdb'
HC, LC, antigen = '8', '6', '2 3'
pdb_HC_protein_minDists, pdb_LC_protein_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)

Importing RMSD data for mature antibody
	INPUTS: apo/CH103_mature_VC 1 121 1 105
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
176 167 6843


#### alignment of pdb data

In [19]:
print(mature_VHC_VLC_resi_names[0][0:2], mature_VHC_VLC_resi_names[1][0:2])
print(pdb_HC_resis[0:2], pdb_LC_resis[0:2])

[GLN1, VAL2] [SER1, TYR2]
[GLN1, VAL2] [TYR2, GLU3]


In [20]:
# Make sure everything is aligned:  
# residues list alignment
pdb_LC_resis_fixed = pdb_LC_resis[:-1] # there's an extra residue at the end here that i want to remove to make the arrays the same length. 
pdb_LC_resis_fixed.insert(0,'SER1')
# check that the mature sequences are aligned. 
for i in range(len(mature_VHC_VLC_resi_names[1])):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])

# minimum distance alignment
pdb_LC_protein_minDists_fixed = np.insert(pdb_LC_protein_minDists[:-1], 0, None) # to account for the added 'ser1' in LC resis. 

ERROR: misalignment 104 GLY105 ARG105


### Naive

In [21]:
# Import RMSD data for the naive antibody now.
print('Importing RMSD data for naive antibody')
system_name='CH103_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 121, 1, 108 # Naive 
naive_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
naive_HC_rmsd_values, naive_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end) #pdb_LC_protein_minDists

Importing RMSD data for naive antibody
	INPUTS: apo/CH103_naive_VC 1 121 1 108


### Alignment

In [22]:
# Heavy chain for CH103 is already aligned. Just save the output. 
print(len(naive_HC_rmsd_values), len(mature_HC_rmsd_values))

121 121


In [23]:
# Light chain for CH103 needs to be aligned. Adding a total of 3 'None' values to mature sequence. Also to PDB mature sequence. 
print(len(naive_LC_rmsd_values), len(mature_LC_rmsd_values))

# add 'None' as gaps for the sequence alignment
x0 = np.insert(mature_LC_rmsd_values, [27], None)
x1 = np.insert(x0, [29,29], None)
mature_LC_rmsd_values_fixed = x1

mature_VLC_resi_names_fixed_ = np.insert(mature_VHC_VLC_resi_names[1], [27], None)
mature_VLC_resi_names_fixed = np.insert(mature_VLC_resi_names_fixed_, [29,29], None)

print(len(naive_LC_rmsd_values), len(mature_LC_rmsd_values_fixed)) #pdb_LC_protein_minDists

# Now fix pdb list
pdb_LC_resis_fixed_fixed_ = np.insert(pdb_LC_resis_fixed, [27], None)
pdb_LC_resis_fixed_fixed = np.insert(pdb_LC_resis_fixed_fixed_, [29,29], None)
x0 = np.insert(pdb_LC_protein_minDists_fixed, [27], None)
x1 = np.insert(x0, [29,29], None)
pdb_LC_protein_minDists_fixed_fixed = x1

108 105
108 108


## Save output for plotting later. 

In [24]:
# HC
data = {
    "pdb_mature_HC_resi":pdb_HC_resis,
    "pdb_mature_HC_protein_minDists":pdb_HC_protein_minDists,
    "naive_HC_resi": naive_VHC_VLC_resi_names[0],
    "naive_HC_rmsd": naive_HC_rmsd_values,
    "mature_HC_resi": mature_VHC_VLC_resi_names[0],
    "mature_HC_rmsd": mature_HC_rmsd_values,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/CH103_HC.csv"
df.to_csv(output_filename, index=False)

# LC
data = {
    "pdb_mature_LC_resi":pdb_LC_resis_fixed_fixed,
    "pdb_mature_LC_protein_minDists":pdb_LC_protein_minDists_fixed_fixed,
    "naive_LC_resi": naive_VHC_VLC_resi_names[1],
    "naive_LC_rmsd": naive_LC_rmsd_values,
    "mature_LC_resi": mature_VLC_resi_names_fixed,
    "mature_LC_rmsd": mature_LC_rmsd_values_fixed,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/CH103_LC.csv"
df.to_csv(output_filename, index=False)

# EPTC112

## Import and Calculate data

### Mature

In [25]:
# import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
print('Importing RMSD data for mature antibody')
system_name='EPTC112_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 121, 1, 110 
mature_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
mature_HC_rmsd_values, mature_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{base_path}structures/8c8t_EPTC112_clean_renumbered.pdb'
HC, LC, antigen = '1', '3', '0'
pdb_HC_protein_minDists, pdb_LC_protein_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)

Importing RMSD data for mature antibody
	INPUTS: apo/EPTC112_mature_VC 1 121 1 110
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
119 108 3277


#### alignment of pdb data

In [26]:
print(mature_VHC_VLC_resi_names[0][0:2], mature_VHC_VLC_resi_names[1][0:2])
print(pdb_HC_resis[0:2], pdb_LC_resis[0:2])

[GLN1, VAL2] [GLN1, SER2]
[VAL2, GLN3] [SER2, VAL3]


In [27]:
# Make sure everything is aligned:  
# residues list alignment
pdb_LC_resis_fixed = pdb_LC_resis[:] 
pdb_LC_resis_fixed.insert(0,'GLN1')
pdb_LC_resis_fixed.insert(len(pdb_LC_resis_fixed),None)
# check that the mature sequences are aligned. 
for i in range(len(pdb_LC_resis_fixed)):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])
# minimum distance alignment
pdb_LC_protein_minDists_fixed_ = np.insert(pdb_LC_protein_minDists, 0, None) # to account for the added 'gln1' in LC resis. 
pdb_LC_protein_minDists_fixed = np.insert(pdb_LC_protein_minDists_fixed_, len(pdb_LC_protein_minDists_fixed_), None) # to account for the missing last residue in LC resis. 

# residues list alignment
pdb_HC_resis_fixed = pdb_HC_resis[:] 
pdb_HC_resis_fixed.insert(0,'GLN1')
pdb_HC_resis_fixed.insert(len(pdb_HC_resis_fixed),None)
# check that the mature sequences are aligned. 
for i in range(len(pdb_HC_resis_fixed)):
    if str(pdb_HC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[0][i]):
        print('ERROR: misalignment',i, pdb_HC_resis_fixed[i], mature_VHC_VLC_resi_names[0][i])
# minimum distance alignment
pdb_HC_protein_minDists_fixed_ = np.insert(pdb_HC_protein_minDists, 0, None) # to account for the added 'gln1' in HC resis. 
pdb_HC_protein_minDists_fixed = np.insert(pdb_HC_protein_minDists_fixed_, len(pdb_HC_protein_minDists_fixed_), None) # to account for the missing last residue in HC resis. 


ERROR: misalignment 109 None LEU110
ERROR: misalignment 120 None SER121


### Naive

In [28]:
# Import RMSD data for the naive antibody now.
print('Importing RMSD data for naive antibody')
system_name='EPTC112_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 121, 1, 110 
naive_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
naive_HC_rmsd_values, naive_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end) #pdb_LC_protein_minDists

Importing RMSD data for naive antibody
	INPUTS: apo/EPTC112_naive_VC 1 121 1 110


## Save output for plotting later. 

In [29]:
# HC
data = {
    "pdb_mature_HC_resi":pdb_HC_resis_fixed,
    "pdb_mature_HC_protein_minDists":pdb_HC_protein_minDists_fixed,
    "naive_HC_resi": naive_VHC_VLC_resi_names[0],
    "naive_HC_rmsd": naive_HC_rmsd_values,
    "mature_HC_resi": mature_VHC_VLC_resi_names[0],
    "mature_HC_rmsd": mature_HC_rmsd_values,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/EPTC112_HC.csv"
df.to_csv(output_filename, index=False)

# LC
data = {
    "pdb_mature_LC_resi":pdb_LC_resis_fixed,
    "pdb_mature_LC_protein_minDists":pdb_LC_protein_minDists_fixed,
    "naive_LC_resi": naive_VHC_VLC_resi_names[1],
    "naive_LC_rmsd": naive_LC_rmsd_values,
    "mature_LC_resi": mature_VHC_VLC_resi_names[1],
    "mature_LC_rmsd": mature_LC_rmsd_values,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/EPTC112_LC.csv"
df.to_csv(output_filename, index=False)

# CH65

## Import and Calculate data

### Mature

In [30]:
pdb_path = f'{base_path}structures/5ugy_CH65_clean_renumbered.pdb'
topology = md.load(pdb_path).topology
print(topology)
table, bonds = topology.to_dataframe()
#print(table)
#print(table[table['chainID'] == 2]) 

HC = '1' 
LC = '2' #'2'
antigen_ = '0' # 6 8 '2 8'
HC_protein_minDists, LC_protein_minDists, HC_glycan_minDists, LC_glycan_minDists = calculate_min_dist_values(pdb_path, HC, LC, antigen_)

<mdtraj.Topology with 7 chains, 763 residues, 11589 atoms, 11754 bonds>


TypeError: calculate_min_dist_values() missing 2 required positional arguments: 'vh_end' and 'vl_end'

In [31]:
# import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
lineage = 'CH65'
print('Importing RMSD data for mature antibody')
system_name=f'{lineage}_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 126, 1, 108
mature_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
mature_HC_rmsd_values, mature_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{base_path}structures/5ugy_{lineage}_clean_renumbered.pdb'
HC, LC, antigen = '1', '2', '0'
pdb_HC_protein_minDists, pdb_LC_protein_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)

Importing RMSD data for mature antibody
	INPUTS: apo/CH65_mature_VC 1 126 1 108
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
221 210 2535


#### alignment of pdb data

In [32]:
print(mature_VHC_VLC_resi_names[0][0:2], mature_VHC_VLC_resi_names[1][0:2])
print(pdb_HC_resis[0:2], pdb_LC_resis[0:2])

[GLU1, VAL2] [GLN1, SER2]
[GLU1, VAL2] [SER2, VAL3]


In [33]:
# Make sure everything is aligned:  
# residues list alignment
pdb_LC_resis_fixed = pdb_LC_resis[:-1]  # remove the last resi. 
pdb_LC_resis_fixed.insert(0,'GLN1')

# check that the mature sequences are aligned. 
for i in range(len(pdb_LC_resis_fixed)):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])
# minimum distance alignment
pdb_LC_protein_minDists_fixed = np.insert(pdb_LC_protein_minDists[:-1], 0, None) # to account for the added 'gln1' in LC resis. 
print(len(pdb_LC_resis_fixed), len(pdb_LC_protein_minDists_fixed))

108 108


### Naive

In [34]:
# Import RMSD data for the naive antibody now.
print('Importing RMSD data for naive antibody')
system_name=f'{lineage}_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 126, 1, 108
naive_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
naive_HC_rmsd_values, naive_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end) #pdb_LC_protein_minDists

Importing RMSD data for naive antibody
	INPUTS: apo/CH65_naive_VC 1 126 1 108


## Save output for plotting later. 

In [35]:
# HC
data = {
    "pdb_mature_HC_resi":pdb_HC_resis,
    "pdb_mature_HC_protein_minDists":pdb_HC_protein_minDists,
    "naive_HC_resi": naive_VHC_VLC_resi_names[0],
    "naive_HC_rmsd": naive_HC_rmsd_values,
    "mature_HC_resi": mature_VHC_VLC_resi_names[0],
    "mature_HC_rmsd": mature_HC_rmsd_values,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/{lineage}_HC.csv"
df.to_csv(output_filename, index=False)

# LC
data = {
    "pdb_mature_LC_resi":pdb_LC_resis_fixed,
    "pdb_mature_LC_protein_minDists":pdb_LC_protein_minDists_fixed,
    "naive_LC_resi": naive_VHC_VLC_resi_names[1],
    "naive_LC_rmsd": naive_LC_rmsd_values,
    "mature_LC_resi": mature_VHC_VLC_resi_names[1],
    "mature_LC_rmsd": mature_LC_rmsd_values,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/{lineage}_LC.csv"
df.to_csv(output_filename, index=False)

# PGT121

In [36]:
lineage = 'PGT121'

## Import and Calculate data

### Mature

In [37]:
# import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
print('Importing RMSD data for mature antibody')
system_name=f'{lineage}_mature_kappa_VC'
HC_start, HC_end, LC_start, LC_end = 1, 132, 1, 105 # Mature 
mature_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
mature_HC_rmsd_values, mature_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{base_path}structures/7uoj_{lineage}_clean_renumbered.pdb'
HC, LC, antigen = '4', '3', '0'
pdb_HC_protein_minDists, pdb_LC_protein_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)

Importing RMSD data for mature antibody
	INPUTS: apo/PGT121_mature_kappa_VC 1 132 1 105
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
133 104 3559


#### alignment of pdb data

In [38]:
print(mature_VHC_VLC_resi_names[0][0:2], mature_VHC_VLC_resi_names[1][0:2])
print(pdb_HC_resis[0:2], pdb_LC_resis[0:2])

print(len(pdb_LC_resis), len(mature_VHC_VLC_resi_names[1]))

[GLN1, MET2] [SER1, ASP2]
[GLN1, MET2] [ASP2, ILE3]
104 105


In [39]:
# Make sure everything is aligned:  
# residues list alignment
pdb_LC_resis_fixed = pdb_LC_resis[:]
pdb_LC_resis_fixed.insert(0,'SER1')
# check that the mature sequences are aligned. 
for i in range(len(mature_VHC_VLC_resi_names[1])):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])

# minimum distance alignment
pdb_LC_protein_minDists_fixed = np.insert(pdb_LC_protein_minDists, 0, None) # to account for the added 'ser1' in LC resis. 

In [40]:
len(pdb_LC_protein_minDists_fixed), len(pdb_LC_resis_fixed)

(105, 105)

### Naive

In [41]:
# Import RMSD data for the naive antibody now.
print('Importing RMSD data for naive antibody')
system_name=f'{lineage}_naive_kappa_VC'
HC_start, HC_end, LC_start, LC_end = 1, 132, 1, 109 # Naive 
naive_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
naive_HC_rmsd_values, naive_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end) #pdb_LC_protein_minDists

Importing RMSD data for naive antibody
	INPUTS: apo/PGT121_naive_kappa_VC 1 132 1 109


### Alignment

In [42]:
# Heavy chain for PGT121 is already aligned. Just save the output. 
print(len(naive_HC_rmsd_values), len(mature_HC_rmsd_values))

132 132


In [43]:
# Light chain for PGT121 needs to be aligned. Adding a total of 4 'None' values to mature sequence. Also to PDB mature sequence. 
print(len(naive_LC_rmsd_values), len(mature_LC_rmsd_values)) 

# add 'None' as gaps for the sequence alignment
x0 = np.insert(mature_LC_rmsd_values, [0,0,0,0 ,0,0,0,0], None)
x1 = np.insert(x0,[98,98],None)
mature_LC_rmsd_values_fixed = x1

x0 = np.insert(mature_VHC_VLC_resi_names[1], [0,0,0,0 ,0,0,0,0], None)
x1 = np.insert(x0,[98,98],None)
mature_VLC_resi_names_fixed = x1

y0 = np.insert(naive_LC_rmsd_values, [9], None)
y1 = np.insert(y0,[65],None)
y2 = np.insert(y1,[68,68],None)
y3 = np.insert(y2,[101,101],None)
naive_LC_rmsd_values_fixed = y3

y0 = np.insert(naive_VHC_VLC_resi_names[1], [9], None)
y1 = np.insert(y0,[65],None)
y2 = np.insert(y1,[68,68],None)
y3 = np.insert(y2,[101,101],None)
naive_VLC_resi_names_fixed = y3

# fix pdb data
x0 = np.insert(pdb_LC_resis_fixed, [0,0,0,0 ,0,0,0,0], None)
x1 = np.insert(x0,[98,98],None)
pdb_LC_resis_fixed_fixed = x1

x0 = np.insert(pdb_LC_protein_minDists_fixed, [0,0,0,0 ,0,0,0,0], None)
x1 = np.insert(x0,[98,98],None)
pdb_LC_protein_minDists_fixed_fixed = x1



109 105


## Save output for plotting later. 

In [44]:
# HC
data = {
    "pdb_mature_HC_resi":pdb_HC_resis,
    "pdb_mature_HC_protein_minDists":pdb_HC_protein_minDists,
    "naive_HC_resi": naive_VHC_VLC_resi_names[0],
    "naive_HC_rmsd": naive_HC_rmsd_values,
    "mature_HC_resi": mature_VHC_VLC_resi_names[0],
    "mature_HC_rmsd": mature_HC_rmsd_values,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/{lineage}_HC.csv"
df.to_csv(output_filename, index=False)

# LC
data = {
    "pdb_mature_LC_resi":pdb_LC_resis_fixed_fixed,
    "pdb_mature_LC_protein_minDists":pdb_LC_protein_minDists_fixed_fixed,
    "naive_LC_resi": naive_VLC_resi_names_fixed,
    "naive_LC_rmsd": naive_LC_rmsd_values_fixed,
    "mature_LC_resi": mature_VLC_resi_names_fixed,
    "mature_LC_rmsd": mature_LC_rmsd_values_fixed,
    }
df = pd.DataFrame(data)
# The 'index=False' argument prevents pandas from writing the DataFrame row indices as a new column in the CSV file
output_filename = f"{base_path}data/to_protein/{lineage}_LC.csv"
df.to_csv(output_filename, index=False)

# PGT124

In [45]:
# import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
print('Importing RMSD data for mature antibody')
lineage = 'PGT124'
system_name=f'{lineage}_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 132, 1, 108 # Mature 
mature_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
rmsd_data = import_data_apo(system_name)
mature_HC_rmsd_values, mature_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{base_path}structures/8d50_PGT124_clean_vc_renumbered.pdb'
HC, LC, antigen = '2', '1', '0'
pdb_HC_protein_minDists, pdb_LC_protein_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)

Importing RMSD data for mature antibody
	INPUTS: apo/PGT124_mature_VC 1 132 1 108
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
131 107 3426


In [46]:
# -------------------------------------------------------
# Align the PDB data to the mature antibody -- some PDB structures are missing the first few residues or last few resis. 
# HC PDB is already aligned.
# Need to add 1 for the HC and LC.
pdb_LC_resis_fixed           = np.insert(pdb_LC_resis,           0,'SER1')
pdb_LC_protein_minDists_fixed = np.insert(pdb_LC_protein_minDists, 0, None)
for i in range(LC_end):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: LC misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])

# Need to add 1 for the HC and LC.
pdb_HC_resis_fixed           = np.insert(pdb_HC_resis,           0,'GLN1')
pdb_HC_protein_minDists_fixed = np.insert(pdb_HC_protein_minDists, 0, None)
for i in range(HC_end):
    if str(pdb_HC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[0][i]):
        print('ERROR: LC misalignment',i, pdb_HC_resis_fixed[i], mature_VHC_VLC_resi_names[0][i])


In [47]:
mature_LC_rmsd_values_fixed=np.insert(mature_LC_rmsd_values,    [3,3,3,3], None)
mature_VLC_resi_names_fixed=np.insert(mature_VHC_VLC_resi_names[1], [3,3,3,3], None)
pdb_LC_resis_fixed_fixed          =np.insert(pdb_LC_resis_fixed,          [3,3,3,3], None)
pdb_LC_protein_minDists_fixed_fixed=np.insert(pdb_LC_protein_minDists_fixed,[3,3,3,3], None)

In [48]:
data={"pdb_mature_LC_resi":pdb_LC_resis_fixed_fixed,
      "pdb_mature_LC_protein_minDists":pdb_LC_protein_minDists_fixed_fixed,
      "mature_LC_resi": mature_VLC_resi_names_fixed,
      "mature_LC_rmsd": mature_LC_rmsd_values_fixed}
df = pd.DataFrame(data)
output_filename = f"{base_path}data/to_protein/PGT124_LC.csv"
df.to_csv(output_filename, index=False)

In [49]:
data={"pdb_mature_HC_resi":pdb_HC_resis_fixed_fixed,
      "pdb_mature_HC_protein_minDists":pdb_HC_protein_minDists_fixed_fixed,
      "mature_HC_resi": mature_VHC_resi_names_fixed,
      "mature_HC_rmsd": mature_HC_rmsd_values_fixed}
df = pd.DataFrame(data)
output_filename = f"{base_path}data/to_protein/PGT124_HC.csv"
df.to_csv(output_filename, index=False)